# Pasado `gpu-hybrid`: batching po `n_splits` na CUDA

Pre pokretanja izaberi **Runtime > Change runtime type > GPU** (poželjno A100), pa **Runtime > Run all**.

Šta ova sveska radi:
1. Kloniraj `gpu-hybrid` sa fork-a i upiše **tri fajla koji još nisu na GitHub-u** (`batched_splits.py`, `test_batched_splits.py`, `measure_split_sweep.py` sa `--split-mode`).
2. **Test korektnosti na CUDA-i**: svaka pojedinačna granica iz batched prolaza mora da se poklapa sa sekvencijalnom petljom.
3. **Sweep `n_splits = 1, 2, 4, 8, 16, 32, 64`** u oba režima (`sequential` naspram `batched`), uz `peak_vram_mb`.
4. **Udeo CPU `gelsd`-a** u batched prolazu na `big` (sumnja: postaje usko grlo jer ostaje na CPU-u).
5. Tabela ubrzanja, grafik i `.zip` za preuzimanje.

Cela sveska traje oko 10 do 15 minuta na A100.

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Uključi GPU: Runtime > Change runtime type > GPU"

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
print("VRAM GiB:", round(VRAM_GIB, 2))

In [ ]:
import os
import shutil
import subprocess

REPO = "/content/Pasado"
URL = "https://github.com/dimitrijepesic/Pasado.git"

# leave the folder first: deleting the current directory breaks git on a re-run
os.chdir("/content")
shutil.rmtree(REPO, ignore_errors=True)
clone = subprocess.run([
    "git", "-c", "http.version=HTTP/1.1", "clone",
    "--branch", "gpu-hybrid", "--single-branch", URL, REPO,
], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(clone.stdout)

head = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
print("HEAD:", head)
os.chdir(REPO)

In [ ]:
%pip install -q scikit-learn

## 0. Novi i izmenjeni fajlovi (još nisu commitovani na GitHub)

In [ ]:
%%writefile /content/Pasado/forward_mode_tensorized_src/batched_splits.py
"""Interval splitting with all sub-intervals in one batched forward pass.

--n-splits divides [0, epsilon] into S pieces and bounds the Lipschitz constant on
each. The original code runs those pieces one after another in a Python loop. The
pieces are independent and have the same shapes, so this module stacks them on a
leading batch axis instead:

    centers     [S, N]        generators     [S, G, N]

and runs one pass of the precise forward analysis over the whole stack. Per split
the arithmetic is the same as in the sequential code (SimpleZono.py and
precise_transformer.py), so the per-split bounds agree with it up to floating
point reordering inside matmul and sum.

Only the precise path for fully connected sigmoid networks is covered. The code
stays outside precise_transformer.py so the sequential path is untouched and can
serve as the reference.

Differences from the sequential code, all numerically irrelevant:
  * The "all lower bounds equal all upper bounds" shortcut in SigmoidZonotope is
    not taken. The general formula gives the same bounds there.
  * Zero rows that the sequential code appends only to align generator counts are
    not materialised when they would be multiplied away.

Memory grows linearly with S (the sequential loop reuses the same buffers), so
`chunk` caps how many splits are processed at once.
"""
import numpy as np
import torch

import precise_transformer as pt


class BatchedDualZonotope:
    """S independent dual zonotopes over N variables with G noise symbols each."""

    __slots__ = ("rc", "rg", "dc", "dg")

    def __init__(self, rc, rg, dc, dg):
        self.rc = rc  # real centers      [S, N]
        self.rg = rg  # real generators   [S, G, N]
        self.dc = dc  # dual centers      [S, N]
        self.dg = dg  # dual generators   [S, G, N]


def extract_affine_layers(net, network_name):
    """[(weight.T, bias), ...] for the hidden layers followed by the output layer.

    The tensors are detached, so later changes to the module (net.to(...), training)
    do not reach them.
    """
    if network_name == "big":
        modules = [net[i] for i in (1, 3, 5, 7, 9)]
    else:
        layers = int(network_name[0])
        names = ["fc1", "fc2"] + (["fc3"] if layers >= 4 else []) + (["fc4"] if layers == 5 else [])
        names.append("fc_final")
        modules = [getattr(net, n) for n in names]
    return [(m.weight.detach().T, m.bias.detach()) for m in modules]


def hazed_input(img, lows, highs):
    """(1 - eps) * img + eps for eps in each [lows[s], highs[s]], as a batched dual zonotope.

    Written out in the operation order of the sequential expression
    (one - zono_eps) * img + zono_eps, so each split gets the same values.
    The two noise symbols are the real one (the interval of eps) and the dual one,
    whose generators are zero because the dual part of eps is the exact value 1.
    """
    mid = (highs + lows) / 2.0
    rad = (highs - lows) / 2.0
    S, N = lows.shape[0], img.shape[0]
    rc = (1.0 - mid).unsqueeze(1) * img + mid.unsqueeze(1)
    rg = img.new_zeros((S, 2, N))
    rg[:, 0, :] = (-rad).unsqueeze(1) * img + rad.unsqueeze(1)
    dc = (-img + 1.0).unsqueeze(0).expand(S, N).contiguous()
    dg = img.new_zeros((S, 2, N))
    return BatchedDualZonotope(rc, rg, dc, dg)


def affine(z, weight_t, bias):
    """AffineDualZonotope(z, weight_t) + bias. The dual part gets no bias."""
    return BatchedDualZonotope(z.rc @ weight_t + bias, z.rg @ weight_t,
                               z.dc @ weight_t, z.dg @ weight_t)


def _bounds(centers, generators):
    radius = generators.abs().sum(dim=1)
    return centers - radius, centers + radius


def _perturbation(S, N, seed, dtype, device):
    """The noise added to coefficient A ([S, N]).

    With a seed the sequential protocol reseeds np.random before every split, so
    every split sees the same stream. The caller reseeds once per forward pass and
    the N values drawn here are shared by all splits. Without a seed each (split,
    neuron) pair draws its own value.
    """
    if seed is not None:
        values = torch.as_tensor(np.random.normal(0, .0001, size=N), dtype=dtype, device=device)
        return values.unsqueeze(0).expand(S, N)
    values = torch.as_tensor(np.random.normal(0, .0001, size=(S, N)), dtype=dtype, device=device)
    return values


def precise_sigmoid(z, seed):
    """PreciseSigmoidDualZonotope for S splits at once. Adds 2N noise symbols."""
    S, G0, N = z.rg.shape
    dtype, device = z.rc.dtype, z.rc.device

    lx, ux = _bounds(z.rc, z.rg)
    ly, uy = _bounds(z.dc, z.dg)

    # SigmoidZonotope on the real part
    s_lb, s_ub = torch.sigmoid(lx), torch.sigmoid(ux)
    lam = torch.min(s_lb * (1 - s_lb), s_ub * (1 - s_ub))
    mu1 = 0.5 * (s_ub + s_lb - lam * (ux + lx))
    mu2 = 0.5 * (s_ub - s_lb - lam * (ux - lx))
    sig_c = lam * z.rc + mu1

    # Planar regression of sigmoid'(x) * y on the 5 x 5 grid of every (split, neuron)
    # box. The boxes are independent, so [S, N] flattens to one batch of S*N.
    lx_f, ux_f, ly_f, uy_f = (t.reshape(-1) for t in (lx, ux, ly, uy))
    grid = pt.get_linspace_batched(lx_f, ux_f, ly_f, uy_f)  # [S*N, 25, 2]
    gx, gy = grid[..., 0], grid[..., 1]
    targets = (1.0 - torch.sigmoid(gx)) * (torch.sigmoid(gx) * gy)
    coef = pt.lin_reg_tensor_batched(grid, targets)  # [S*N, 3], intercept first
    As = coef[:, 1] + _perturbation(S, N, seed, dtype, device).reshape(-1)
    Bs = coef[:, 2]
    Cs = coef[:, 0]
    assert bool(((As > 0) | (As < 0)).all())

    # worst-case approximation error, same three checks as compute_max_error
    abc = torch.stack((As, Bs, Cs), dim=1)
    corners = pt.check_corners_tensor(abc, lx_f, ux_f, ly_f, uy_f)
    err_ly, err_uy = pt.check_nonlinear_boundary_tensor(lx_f, ux_f, ly_f, uy_f, abc)
    errors = torch.maximum(torch.maximum(corners, err_ly), err_uy).reshape(S, N)

    As, Bs, Cs = As.reshape(S, N), Bs.reshape(S, N), Cs.reshape(S, N)
    dual_c = As * z.rc + Bs * z.dc + Cs

    # Generator layout per split, G0 + 2N rows:
    #   real: [lam * rg | diag(mu2) | 0]      dual: [A * rg + B * dg | 0 | diag(errors)]
    new_rg = z.rg.new_zeros((S, G0 + 2 * N, N))
    new_rg[:, :G0, :] = lam.unsqueeze(1) * z.rg
    new_rg[:, G0:G0 + N, :].diagonal(dim1=1, dim2=2).copy_(mu2)

    new_dg = z.rg.new_zeros((S, G0 + 2 * N, N))
    new_dg[:, :G0, :] = As.unsqueeze(1) * z.rg + Bs.unsqueeze(1) * z.dg
    new_dg[:, G0 + N:, :].diagonal(dim1=1, dim2=2).copy_(errors)

    return BatchedDualZonotope(sig_c, new_rg, dual_c, new_dg)


def _forward_chunk(layers, img, lows, highs, seed):
    z = hazed_input(img, lows, highs)
    if seed is not None:
        np.random.seed(seed)
    for weight_t, bias in layers[:-1]:
        z = precise_sigmoid(affine(z, weight_t, bias), seed)
    weight_t, bias = layers[-1]
    z = affine(z, weight_t, bias)
    lb, ub = _bounds(z.dc, z.dg)
    return torch.maximum(lb.abs(), ub.abs()).amax(dim=1)  # [S]


def split_bounds(layers, img, epsilon, n_splits, seed=None, chunk=None):
    """Lipschitz bound of every sub-interval of [0, epsilon], as an [n_splits] tensor.

    layers comes from extract_affine_layers. seed matches the sequential
    protocol (np.random reseeded before each split); None draws fresh noise.
    chunk limits how many splits go through the network together; None means all.
    The maximum of the result is the bound for the whole interval.
    """
    dtype, device = img.dtype, img.device
    delta = epsilon / n_splits
    # floats first, as in the sequential code, so the endpoints are identical
    lows_f = [i * delta for i in range(n_splits)]
    highs_f = [low + delta for low in lows_f]
    lows = torch.tensor(lows_f, dtype=dtype, device=device)
    highs = torch.tensor(highs_f, dtype=dtype, device=device)

    chunk = n_splits if chunk is None else max(1, min(chunk, n_splits))
    with torch.no_grad():
        parts = [_forward_chunk(layers, img, lows[i:i + chunk], highs[i:i + chunk], seed)
                 for i in range(0, n_splits, chunk)]
    return torch.cat(parts)


def lipschitz_bound(layers, img, epsilon, n_splits, seed=None, chunk=None):
    """max over the splits, as a Python float."""
    return split_bounds(layers, img, epsilon, n_splits, seed, chunk).max().item()


In [ ]:
%%writefile /content/Pasado/experiments/test_batched_splits.py
"""Tests for forward_mode_tensorized_src/batched_splits.py against the sequential code.

The reference is the original loop over sub-intervals (DualIntervalTensor ->
HyperDualIntervalToDualZonotope -> PreciseSigmoidDualZonotope), run once per split
with np.random reseeded each time. The batched pass must reproduce every per-split
bound, not just their maximum.

Run (CPU, float64):
  python experiments/test_batched_splits.py
  python experiments/test_batched_splits.py --device cuda
"""
import argparse
import copy
import os
import sys

import numpy as np
import torch

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, os.path.join(ROOT, "forward_mode_tensorized_src"))
sys.path.insert(0, os.path.join(ROOT, "Section_5_4"))
sys.path.insert(0, HERE)

import precise_transformer as pt
import batched_splits as bs
from Duals import DualIntervalTensor
from SimpleZono import HyperDualIntervalToDualZonotope
from measure_split_sweep import build_forward_precise, load_network_and_images
from check_sweep_soundness import sampled_max_derivative

# float64 reordering noise is ~1e-16 per operation; 1e-10 leaves room for the 4 to 5
# layers of accumulation but is far below any difference between two real bounds.
RTOL = 1e-10
EPS_SCHEDULE = [10 ** (-k / 4) * 2 for k in range(2, 18)]

failures = []


def check(name, ok, detail=""):
    print(f"  [{'PASS' if ok else 'FAIL'}] {name} {detail}")
    if not ok:
        failures.append(name)


def sequential_split_bounds(forward_fn, img, epsilon, n_splits, seed, device, dtype):
    """Per-split bounds from the original loop."""
    delta = epsilon / n_splits
    one = torch.tensor([1.0], dtype=dtype, device=device)
    out = []
    with torch.no_grad():
        for i in range(n_splits):
            l_val = i * delta
            interval = DualIntervalTensor(
                real_l=torch.tensor([l_val], dtype=dtype, device=device),
                real_u=torch.tensor([l_val + delta], dtype=dtype, device=device))
            interval.e1_l[0] = 1
            interval.e1_u[0] = 1
            zono_eps = HyperDualIntervalToDualZonotope(interval)
            hazed = (one - zono_eps) * img + zono_eps
            np.random.seed(seed)
            output = forward_fn(hazed)
            out.append(torch.max(torch.maximum(output.dual.get_lb().abs(),
                                               output.dual.get_ub().abs())).item())
    return torch.tensor(out, dtype=torch.float64)


def rel_diff(a, b):
    a, b = a.double().cpu(), b.double().cpu()
    return ((a - b).abs() / b.abs().clamp_min(1e-300)).max().item()


def test_matches_sequential(device, dtype, networks, split_values, eps_indices, num_images):
    print("batched == sequential, per split")
    for network in networks:
        net, images = load_network_and_images(network, device, dtype, num_images)
        forward_fn = build_forward_precise(pt, net, network)
        layers = bs.extract_affine_layers(net, network)
        worst = 0.0
        for image_id, img in images:
            for eps_idx in eps_indices:
                for n_splits in split_values:
                    eps = EPS_SCHEDULE[eps_idx]
                    ref = sequential_split_bounds(forward_fn, img, eps, n_splits, 7, device, dtype)
                    got = bs.split_bounds(layers, img, eps, n_splits, seed=7)
                    worst = max(worst, rel_diff(got, ref))
        check(f"{network}: all per-split bounds", worst <= RTOL,
              f"(max rel diff {worst:.2e}, splits {split_values}, eps {eps_indices}, "
              f"{len(images)} images)")


def test_chunking(device, dtype, network="3layer"):
    print("chunking does not change the result")
    net, images = load_network_and_images(network, device, dtype, 1)
    layers = bs.extract_affine_layers(net, network)
    img = images[0][1]
    eps = EPS_SCHEDULE[4]
    full = bs.split_bounds(layers, img, eps, 8, seed=3)
    for chunk in (1, 3, 8, 100):
        part = bs.split_bounds(layers, img, eps, 8, seed=3, chunk=chunk)
        check(f"chunk={chunk}", rel_diff(part, full) <= RTOL, f"({rel_diff(part, full):.2e})")


def test_split_one_is_plain_bound(device, dtype, network="3layer"):
    print("n_splits=1 equals one sequential forward pass")
    net, images = load_network_and_images(network, device, dtype, 1)
    forward_fn = build_forward_precise(pt, net, network)
    layers = bs.extract_affine_layers(net, network)
    img = images[0][1]
    eps = EPS_SCHEDULE[0]
    ref = sequential_split_bounds(forward_fn, img, eps, 1, 5, device, dtype)
    got = bs.split_bounds(layers, img, eps, 1, seed=5)
    check("single split", rel_diff(got, ref) <= RTOL)


def test_unseeded_is_sound(device, dtype, network="3layer"):
    """Without a seed there is no reference to match, so check soundness instead:
    the bound must not fall below a sampled maximum of the true derivative."""
    print("unseeded noise: bound stays above the sampled autograd maximum")
    net, images = load_network_and_images(network, device, dtype, 2)
    layers = bs.extract_affine_layers(net, network)
    # a copy: Module.to moves parameters in place, which would pull the bias
    # tensors in `layers` off the device while their weights stay on it
    net64 = copy.deepcopy(net).to("cpu", torch.float64)
    for image_id, img in images:
        for eps_idx in (0, 8):
            eps = EPS_SCHEDULE[eps_idx]
            sampled = sampled_max_derivative(net64, img.cpu().double(), eps, 1001)
            for n_splits in (1, 16):
                lc = bs.lipschitz_bound(layers, img, eps, n_splits, seed=None)
                check(f"image {image_id} eps_idx {eps_idx} n={n_splits}", lc >= sampled,
                      f"(lc={lc:.4f}, sampled={sampled:.4f})")


def test_splits_are_independent(device, dtype, network="3layer"):
    """Bound for a split must not depend on which other splits share its batch."""
    print("per-split bound is independent of the other splits in the batch")
    net, images = load_network_and_images(network, device, dtype, 1)
    layers = bs.extract_affine_layers(net, network)
    img = images[0][1]
    eps = EPS_SCHEDULE[2]
    all16 = bs.split_bounds(layers, img, eps, 16, seed=1)
    # the same sub-intervals, evaluated as 4 batches of 4
    parts = bs.split_bounds(layers, img, eps, 16, seed=1, chunk=4)
    check("16 splits vs 4 chunks of 4", rel_diff(parts, all16) <= RTOL)


def main():
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--device", default="cpu")
    ap.add_argument("--dtype", choices=["float32", "float64"], default="float64")
    ap.add_argument("--quick", action="store_true", help="3layer only, fewer cases")
    args = ap.parse_args()

    device = torch.device(args.device)
    dtype = getattr(torch, args.dtype)
    torch.set_default_dtype(dtype)
    pt.USE_VECTORIZED_PRECISE = True
    pt.USE_VECTORIZED_BOUNDARY = True
    pt.USE_BATCHED_LSTSQ = True
    pt.USE_BATCHED_GRID = True
    pt.USE_REAL_CUBIC = True

    if args.quick:
        test_matches_sequential(device, dtype, ["3layer"], [1, 4], [0, 8], 1)
    else:
        test_matches_sequential(device, dtype, ["3layer", "4layer", "5layer"],
                                [1, 2, 3, 8], [0, 8, 15], 3)
        test_matches_sequential(device, dtype, ["big"], [1, 4], [0, 8], 1)
    test_chunking(device, dtype)
    test_split_one_is_plain_bound(device, dtype)
    test_splits_are_independent(device, dtype)
    test_unseeded_is_sound(device, dtype)

    print()
    if failures:
        print(f"FAILED: {len(failures)} check(s): {failures}")
        return 1
    print("ALL CHECKS PASSED")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /content/Pasado/experiments/measure_split_sweep.py
"""Measures the precision vs. runtime trade-off of interval splitting (--n-splits).

For each network, image, epsilon index, and n_splits value (default: 1, 2, 4, 8, 16),
this script divides [0, epsilon] into n_splits equal sub-intervals:
  [i * delta, (i + 1) * delta],  delta = epsilon / n_splits,
evaluates the precise forward pass on each sub-interval, and takes the maximum
local Lipschitz bound across the sub-intervals.

It records:
  - lc_precise:           bound for (network, image, epsilon, n_splits)
  - lc_split1:            baseline bound at n_splits = 1
  - bound_reduction_pct:  (lc_split1 - lc_precise) / lc_split1 * 100
  - tightening_factor:    lc_split1 / lc_precise
  - time_median_ms:       median wall-clock time (ms) across measured repeats
  - time_min_ms, time_max_ms
  - time_multiplier:      time_median_ms / time_split1_median_ms

Run:
  python experiments/measure_split_sweep.py --device cuda --networks 3layer big --out split_sweep.csv
  python experiments/measure_split_sweep.py --device mps --networks 3layer
  python experiments/measure_split_sweep.py --device cuda --split-mode batched --splits 1 8 32 64

--split-mode batched runs all sub-intervals in one forward pass (batched_splits.py)
instead of a Python loop. --split-chunk caps how many go through together, which
bounds memory (it grows linearly with the number of splits in a batch).
"""
import argparse
import csv
import os
import statistics
import sys
import time

import numpy as np
import torch

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, os.path.join(ROOT, "forward_mode_tensorized_src"))
sys.path.insert(0, os.path.join(ROOT, "Section_5_4"))

CSV_COLUMNS = [
    "network",
    "device",
    "dtype",
    "image_id",
    "epsilon_index",
    "epsilon",
    "n_splits",
    "lc_precise",
    "lc_split1",
    "bound_reduction_pct",
    "tightening_factor",
    "time_median_ms",
    "time_min_ms",
    "time_max_ms",
    "time_split1_median_ms",
    "time_multiplier",
    "per_split_ms",
    "peak_vram_mb",
    "split_mode",
    "split_chunk",
]


def positive_int(value):
    parsed = int(value)
    if parsed < 1:
        raise argparse.ArgumentTypeError("must be at least 1")
    return parsed


def non_negative_int(value):
    parsed = int(value)
    if parsed < 0:
        raise argparse.ArgumentTypeError("must be at least 0")
    return parsed


def numpy_seed(value):
    parsed = int(value)
    if not 0 <= parsed < 2 ** 32:
        raise argparse.ArgumentTypeError("must be between 0 and 2**32 - 1")
    return parsed


def parse_args():
    ap = argparse.ArgumentParser(
        description="Measure precision vs. runtime trade-off across --n-splits values"
    )
    ap.add_argument(
        "--networks",
        "--network",
        dest="networks",
        nargs="+",
        choices=["3layer", "4layer", "5layer", "big"],
        default=["3layer", "big"],
        help="networks to evaluate; default: 3layer big",
    )
    ap.add_argument(
        "--device",
        default=os.environ.get("PASADO_DEVICE", "cuda"),
        help="torch device (cuda, mps, cpu); default: PASADO_DEVICE or cuda",
    )
    ap.add_argument(
        "--dtype",
        choices=["float32", "float64"],
        default=None,
        help="floating-point dtype; default: float32 on mps, float64 otherwise",
    )
    ap.add_argument(
        "--splits",
        type=positive_int,
        nargs="+",
        default=[1, 2, 4, 8, 16],
        help="list of n_splits values to evaluate; default: 1 2 4 8 16",
    )
    ap.add_argument(
        "--num-images",
        type=positive_int,
        default=3,
        help="number of correctly classified test images; default: 3",
    )
    ap.add_argument(
        "--eps-indices",
        type=int,
        nargs="+",
        default=[0, 8, 15],
        help="indices into the 16-epsilon schedule; default: 0 8 15",
    )
    ap.add_argument("--seed", type=numpy_seed, default=7)
    ap.add_argument(
        "--warmup",
        type=non_negative_int,
        default=2,
        help="number of unmeasured warm-up runs per configuration; default: 2",
    )
    ap.add_argument(
        "--repeats",
        type=positive_int,
        default=3,
        help="number of timed repeats per (image, epsilon, n_splits); default: 3",
    )
    ap.add_argument(
        "--split-mode",
        choices=["sequential", "batched"],
        default="sequential",
        help="sequential: one forward pass per sub-interval (original); "
             "batched: all sub-intervals in one pass; default: sequential",
    )
    ap.add_argument(
        "--split-chunk",
        type=positive_int,
        default=None,
        help="batched mode only: sub-intervals per forward pass; default: all at once",
    )
    ap.add_argument(
        "--real-cubic",
        dest="real_cubic",
        action="store_true",
        default=True,
        help="use the real Viete cubic root solver (default)",
    )
    ap.add_argument(
        "--no-real-cubic",
        dest="real_cubic",
        action="store_false",
        help="use the legacy complex Cardano cubic root solver",
    )
    ap.add_argument(
        "--out",
        default=None,
        help="optional path to write CSV measurements",
    )
    args = ap.parse_args()

    if args.split_chunk is not None and args.split_mode != "batched":
        ap.error("--split-chunk needs --split-mode batched")
    if 1 not in args.splits:
        args.splits = [1] + list(args.splits)
    if len(set(args.eps_indices)) != len(args.eps_indices):
        ap.error("--eps-indices must not contain duplicates")
    invalid_eps = [i for i in args.eps_indices if not 0 <= i < 16]
    if invalid_eps:
        ap.error(f"--eps-indices entries must be between 0 and 15: {invalid_eps}")

    try:
        device = torch.device(args.device)
    except (RuntimeError, ValueError) as error:
        ap.error(f"invalid --device {args.device!r}: {error}")
    if device.type == "cuda" and not torch.cuda.is_available():
        sys.exit("--device cuda was requested but CUDA is not available")
    if device.type == "mps" and not torch.backends.mps.is_available():
        sys.exit("--device mps was requested but MPS is not available")

    default_dtype = "float32" if device.type == "mps" else "float64"
    dtype_name = args.dtype or default_dtype
    if device.type == "mps" and dtype_name == "float64":
        ap.error("MPS does not support float64; use --dtype float32")

    return args, device, dtype_name, getattr(torch, dtype_name)


def sync_device(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        torch.mps.synchronize()


def stamp(device):
    sync_device(device)
    return time.perf_counter()


def load_network_and_images(network_name, device, dtype, num_images):
    import torchvision
    import torchvision.transforms as transforms
    from model import FCN, FCNBig

    section_dir = os.path.join(ROOT, "Section_5_4")
    if network_name == "3layer":
        net = FCN(3)
    elif network_name == "4layer":
        net = FCN(4)
    elif network_name == "5layer":
        net = FCN(5)
    elif network_name == "big":
        net = FCNBig()
    else:
        raise ValueError(f"unknown network: {network_name}")

    weights_path = os.path.join(section_dir, "trained", f"model_{network_name}.pth")
    net.load_state_dict(torch.load(weights_path, map_location="cpu"))
    net = net.eval().to(dtype=dtype, device=device)

    indices_path = os.path.join(section_dir, "trained", f"indices_{network_name}.pth")
    correct_indices = {
        int(index)
        for index in torch.load(indices_path, map_location="cpu")
    }
    if num_images > len(correct_indices):
        raise RuntimeError(
            f"--num-images {num_images} exceeds the {len(correct_indices)} "
            f"available correctly classified images for {network_name}"
        )

    testset = torchvision.datasets.MNIST(
        root=os.path.join(section_dir, "MNIST_Data"),
        train=False,
        download=True,
        transform=transforms.ToTensor(),
    )
    selected = []
    for idx in range(len(testset)):
        if idx in correct_indices:
            image_tensor, _ = testset[idx]
            selected.append((idx, image_tensor.flatten().to(dtype=dtype, device=device)))
            if len(selected) >= num_images:
                break
    return net, selected


def build_forward_precise(pt_mod, net, network_name):
    if network_name == "big":
        def forward_zono_precise(x):
            x = pt_mod.PreciseSigmoidDualZonotope(pt_mod.AffineDualZonotope(x, net[1].weight.T) + net[1].bias)
            x = pt_mod.PreciseSigmoidDualZonotope(pt_mod.AffineDualZonotope(x, net[3].weight.T) + net[3].bias)
            x = pt_mod.PreciseSigmoidDualZonotope(pt_mod.AffineDualZonotope(x, net[5].weight.T) + net[5].bias)
            x = pt_mod.PreciseSigmoidDualZonotope(pt_mod.AffineDualZonotope(x, net[7].weight.T) + net[7].bias)
            x = pt_mod.AffineDualZonotope(x, net[9].weight.T) + net[9].bias
            return x
        return forward_zono_precise

    layers = int(network_name[0])

    def forward_zono_precise(x):
        x = pt_mod.PreciseSigmoidDualZonotope(pt_mod.AffineDualZonotope(x, net.fc1.weight.T) + net.fc1.bias.data)
        x = pt_mod.PreciseSigmoidDualZonotope(pt_mod.AffineDualZonotope(x, net.fc2.weight.T) + net.fc2.bias.data)
        if layers >= 4:
            x = pt_mod.PreciseSigmoidDualZonotope(pt_mod.AffineDualZonotope(x, net.fc3.weight.T) + net.fc3.bias.data)
        if layers == 5:
            x = pt_mod.PreciseSigmoidDualZonotope(pt_mod.AffineDualZonotope(x, net.fc4.weight.T) + net.fc4.bias.data)
        x = pt_mod.AffineDualZonotope(x, net.fc_final.weight.T) + net.fc_final.bias.data
        return x

    return forward_zono_precise


def eval_splits_once(forward_fn, image, eps_value, n_splits, seed, device, dtype):
    from Duals import DualIntervalTensor
    from SimpleZono import HyperDualIntervalToDualZonotope

    delta = eps_value / n_splits
    one = torch.tensor([1.0], dtype=dtype, device=device)
    sub_bounds = []
    with torch.no_grad():
        for split_idx in range(n_splits):
            l_val = split_idx * delta
            r_val = l_val + delta
            interval_eps = DualIntervalTensor(
                real_l=torch.tensor([l_val], dtype=dtype, device=device),
                real_u=torch.tensor([r_val], dtype=dtype, device=device),
            )
            interval_eps.e1_l[0] = 1
            interval_eps.e1_u[0] = 1
            zono_eps = HyperDualIntervalToDualZonotope(interval_eps)
            hazed_zono = (one - zono_eps) * image + zono_eps
            np.random.seed(seed)
            output = forward_fn(hazed_zono)
            bound_val = torch.max(
                torch.maximum(torch.abs(output.dual.get_lb()), torch.abs(output.dual.get_ub()))
            ).item()
            sub_bounds.append(bound_val)
    return max(sub_bounds)


def main():
    args, device, dtype_name, dtype = parse_args()

    import precise_transformer as pt
    import batched_splits

    torch.set_default_dtype(dtype)
    pt.USE_VECTORIZED_PRECISE = True
    pt.USE_VECTORIZED_BOUNDARY = True
    pt.USE_BATCHED_LSTSQ = True
    pt.USE_BATCHED_GRID = True
    pt.USE_REAL_CUBIC = args.real_cubic

    eps_schedule = [10 ** (-k / 4) * 2 for k in range(2, 18)]
    all_rows = []

    print(
        f"Split sweep | device={device} | dtype={dtype_name} | real_cubic={args.real_cubic} | "
        f"splits={args.splits} | num_images={args.num_images} | eps_indices={args.eps_indices} | "
        f"warmup={args.warmup} | repeats={args.repeats} | split_mode={args.split_mode} | "
        f"split_chunk={args.split_chunk}"
    )

    for network_name in args.networks:
        net, selected_images = load_network_and_images(
            network_name, device, dtype, args.num_images
        )
        forward_fn = build_forward_precise(pt, net, network_name)
        layers = batched_splits.extract_affine_layers(net, network_name)

        def run_splits(image, eps_value, n_splits):
            if args.split_mode == "batched":
                return batched_splits.lipschitz_bound(
                    layers, image, eps_value, n_splits, args.seed, args.split_chunk)
            return eval_splits_once(forward_fn, image, eps_value, n_splits, args.seed, device, dtype)

        # Warm-up passes on the first image and first epsilon. Batched mode also
        # warms up at the largest split count, since its shapes depend on it.
        _, first_img = selected_images[0]
        first_eps = eps_schedule[args.eps_indices[0]]
        for _ in range(args.warmup):
            run_splits(first_img, first_eps, 1)
            if args.split_mode == "batched":
                run_splits(first_img, first_eps, max(args.splits))

        print(f"\n==================== Network: {network_name} ({device}, {dtype_name}) ====================")

        for eps_idx in args.eps_indices:
            eps_val = eps_schedule[eps_idx]
            print(f"\n--- eps_idx={eps_idx} (epsilon={eps_val:.6f}), averaged over {len(selected_images)} images ---")
            print(
                f"{'n_splits':>8s} | {'mean_lc_precise':>16s} | {'reduction_pct':>13s} | "
                f"{'tightening':>11s} | {'time_med_ms':>12s} | {'time_mult':>10s} | "
                f"{'ms_per_split':>12s} | {'peak_vram_mb':>12s}"
            )
            print("-" * 113)

            per_image_split1 = {}
            for image_id, image in selected_images:
                for n_splits in args.splits:
                    times_ms = []
                    peak_mbs = []
                    lc_val = None
                    for _ in range(args.repeats):
                        if device.type == "cuda":
                            torch.cuda.reset_peak_memory_stats(device)
                        t0 = stamp(device)
                        lc_val = run_splits(image, eps_val, n_splits)
                        t1 = stamp(device)
                        times_ms.append((t1 - t0) * 1000.0)
                        if device.type == "cuda":
                            peak_mbs.append(torch.cuda.max_memory_allocated(device) / (1024.0 ** 2))
                        else:
                            peak_mbs.append(0.0)

                    t_med = statistics.median(times_ms)
                    t_min = min(times_ms)
                    t_max = max(times_ms)
                    peak_mb = max(peak_mbs)

                    if n_splits == 1:
                        per_image_split1[image_id] = (lc_val, t_med)

                    lc_1, t_1 = per_image_split1[image_id]
                    red_pct = (lc_1 - lc_val) / max(abs(lc_1), 1e-300) * 100.0
                    tight_factor = lc_1 / max(abs(lc_val), 1e-300)
                    t_mult = t_med / max(t_1, 1e-300)

                    all_rows.append(
                        {
                            "network": network_name,
                            "device": str(device),
                            "dtype": dtype_name,
                            "image_id": image_id,
                            "epsilon_index": eps_idx,
                            "epsilon": eps_val,
                            "n_splits": n_splits,
                            "lc_precise": lc_val,
                            "lc_split1": lc_1,
                            "bound_reduction_pct": red_pct,
                            "tightening_factor": tight_factor,
                            "time_median_ms": t_med,
                            "time_min_ms": t_min,
                            "time_max_ms": t_max,
                            "time_split1_median_ms": t_1,
                            "time_multiplier": t_mult,
                            "per_split_ms": t_med / n_splits,
                            "peak_vram_mb": peak_mb,
                            "split_mode": args.split_mode,
                            "split_chunk": args.split_chunk if args.split_chunk else "",
                        }
                    )

            for n_splits in args.splits:
                sub = [
                    r
                    for r in all_rows
                    if r["network"] == network_name
                    and r["epsilon_index"] == eps_idx
                    and r["n_splits"] == n_splits
                ]
                mean_lc = statistics.mean(r["lc_precise"] for r in sub)
                mean_lc1 = statistics.mean(r["lc_split1"] for r in sub)
                red_pct = (mean_lc1 - mean_lc) / max(abs(mean_lc1), 1e-300) * 100.0
                tight = mean_lc1 / max(abs(mean_lc), 1e-300)
                med_time = statistics.mean(r["time_median_ms"] for r in sub)
                med_time1 = statistics.mean(r["time_split1_median_ms"] for r in sub)
                t_mult = med_time / max(med_time1, 1e-300)
                per_split = med_time / n_splits
                peak_vram = max(r["peak_vram_mb"] for r in sub)
                print(
                    f"{n_splits:8d} | {mean_lc:16.6f} | {red_pct:12.2f}% | "
                    f"{tight:10.2f}x | {med_time:12.2f} | {t_mult:9.2f}x | "
                    f"{per_split:12.2f} | {peak_vram:10.2f}MB"
                )

    if args.out:
        out_dir = os.path.dirname(os.path.abspath(args.out))
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
        with open(args.out, "w", newline="") as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames=CSV_COLUMNS)
            writer.writeheader()
            writer.writerows(all_rows)
        print(f"\nWrote {len(all_rows)} rows to {args.out}")


if __name__ == "__main__":
    main()


## 1. Test korektnosti na CUDA-i

Poredi **svaku pojedinačnu** granicu sub-intervala iz batched prolaza sa sekvencijalnom petljom (`float64`, seed 7) na `3layer`, `4layer`, `5layer` i `big`, uz proveru chunking-a, nezavisnosti splitova i sigurnosti naspram autograd-a.
Očekivano: `ALL CHECKS PASSED`, razlika reda `1e-14`.

In [ ]:
cmd = '''
set -o pipefail
PYTHONUNBUFFERED=1 python experiments/test_batched_splits.py --device cuda 2>&1 | tee /content/test_batched_splits_cuda.log
'''
res = subprocess.run(["bash", "-lc", cmd], cwd=REPO)
assert res.returncode == 0, "test_batched_splits.py nije prosao; ne nastavljaj dalje, posalji log"

## 2. Sweep: `sequential` naspram `batched`

- `sequential`: stara petlja, jedan forward po sub-intervalu.
- `batched`: svi sub-intervali u jednom prolazu. Memorija raste linearno sa brojem splitova u jednom prolazu (oko 0.5 GB po splitu na `big`), pa se za `big` bira `--split-chunk` prema veličini GPU-a.

`eps_idx = 0` je velika perturbacija (`ε ≈ 0.632`), `eps_idx = 8` je mala (`ε ≈ 0.0063`).

In [ ]:
import math

# about 0.5 GB per split on big in float64 (measured at n_splits=1); keep to 60% of VRAM
budget_gib = 0.6 * VRAM_GIB
BIG_CHUNK = min(64, 2 ** int(math.log2(max(1, budget_gib / 0.6))))
print(f"VRAM {VRAM_GIB:.1f} GiB -> --split-chunk {BIG_CHUNK} for big")

COMMON = "--device cuda --dtype float64 --splits 1 2 4 8 16 32 64 --num-images 3 --eps-indices 0 8 --seed 7 --warmup 2 --repeats 3"

def run(cmd_args, prefix):
    cmd = f'''
set -o pipefail
PYTHONUNBUFFERED=1 python experiments/measure_split_sweep.py {cmd_args} --out {prefix}.csv 2>&1 | tee {prefix}.log
'''
    res = subprocess.run(["bash", "-lc", cmd], cwd=REPO)
    assert res.returncode == 0, f"pad: {prefix}"

### 2a. sequential (sve 4 mreže)

In [ ]:
run(f"--networks 3layer 4layer 5layer big {COMMON} --split-mode sequential", "/content/sweep_sequential")

### 2b. batched, male mreže (svi splitovi odjednom)

In [ ]:
run(f"--networks 3layer 4layer 5layer {COMMON} --split-mode batched", "/content/sweep_batched_small")

### 2c. batched, `big` (sa `--split-chunk` prema VRAM-u)

In [ ]:
run(f"--networks big {COMMON} --split-mode batched --split-chunk {BIG_CHUNK}", "/content/sweep_batched_big")

## 3. Poređenje: isti `lc_precise`? koliko je brže? koliko memorije?

`max_rel_diff_lc` mora biti oko `1e-14`. Ako nije, batched i sekvencijalni ne daju istu granicu i brzina ne vredi ništa.

In [ ]:
import pandas as pd
from IPython.display import display

seq = pd.read_csv("/content/sweep_sequential.csv")
bat = pd.concat([pd.read_csv("/content/sweep_batched_small.csv"),
                 pd.read_csv("/content/sweep_batched_big.csv")])

key = ["network", "image_id", "epsilon_index", "n_splits"]
m = seq.merge(bat, on=key, suffixes=("_seq", "_bat"))
assert len(m) == len(seq) == len(bat), (len(m), len(seq), len(bat))
m["rel_diff_lc"] = (m["lc_precise_seq"] - m["lc_precise_bat"]).abs() / m["lc_precise_seq"].abs()

summary = (
    m.groupby(["network", "epsilon_index", "n_splits"], as_index=False)
    .agg(lc=("lc_precise_bat", "mean"),
         max_rel_diff_lc=("rel_diff_lc", "max"),
         t_seq_ms=("time_median_ms_seq", "mean"),
         t_bat_ms=("time_median_ms_bat", "mean"),
         vram_seq_mb=("peak_vram_mb_seq", "max"),
         vram_bat_mb=("peak_vram_mb_bat", "max"))
)
summary["speedup"] = summary["t_seq_ms"] / summary["t_bat_ms"]
pd.set_option("display.width", 200)
display(summary.round(4))

worst = summary["max_rel_diff_lc"].max()
print(f"\nNajveca relativna razlika lc_precise (batched naspram sequential): {worst:.2e}")
print("OK" if worst < 1e-9 else "PROBLEM: granice se razlikuju")

## 4. Udeo CPU `gelsd`-a u batched prolazu na `big`

`gelsd` postoji samo na CPU-u, pa se u batched prolazu radi `S*N` malih problema na CPU-u (D2H, `gelsd`, H2D). Merimo koliki deo batched vremena to nosi.

In [ ]:
import sys, time, statistics
for p in (f"{REPO}/forward_mode_tensorized_src", f"{REPO}/Section_5_4", f"{REPO}/experiments"):
    if p not in sys.path:
        sys.path.insert(0, p)

import precise_transformer as pt
import batched_splits as bsp
from measure_split_sweep import load_network_and_images

torch.set_default_dtype(torch.float64)
pt.USE_REAL_CUBIC = True
device = torch.device("cuda")

net, imgs = load_network_and_images("big", device, torch.float64, 1)
layers = bsp.extract_affine_layers(net, "big")
img = imgs[0][1]
eps = 2 * 10 ** (-2 / 4)

orig_lstsq = pt.lin_reg_tensor_batched
acc = {"t": 0.0}

def timed_lstsq(x, z):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = orig_lstsq(x, z)
    torch.cuda.synchronize()
    acc["t"] += time.perf_counter() - t0
    return out

pt.lin_reg_tensor_batched = timed_lstsq
rows = []
try:
    for S in (1, 8, 32, 64):
        chunk = min(S, BIG_CHUNK)
        for _ in range(2):
            bsp.lipschitz_bound(layers, img, eps, S, seed=7, chunk=chunk)
        totals, lstsqs = [], []
        for _ in range(5):
            acc["t"] = 0.0
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            bsp.lipschitz_bound(layers, img, eps, S, seed=7, chunk=chunk)
            torch.cuda.synchronize()
            totals.append((time.perf_counter() - t0) * 1000)
            lstsqs.append(acc["t"] * 1000)
        t, l = statistics.median(totals), statistics.median(lstsqs)
        rows.append({"n_splits": S, "chunk": chunk, "total_ms": t, "lstsq_cpu_ms": l,
                     "lstsq_share_%": 100 * l / t, "rest_ms": t - l})
finally:
    pt.lin_reg_tensor_batched = orig_lstsq

import pandas as pd
display(pd.DataFrame(rows).round(2))

## 5. Grafik: vreme i memorija naspram `n_splits`

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(18, 7), dpi=110)
for col, net_name in enumerate(["3layer", "4layer", "5layer", "big"]):
    sub = summary[(summary["network"] == net_name) & (summary["epsilon_index"] == 0)]
    ax = axes[0, col]
    ax.plot(sub["n_splits"], sub["t_seq_ms"], "o-", label="sequential", color="#d62728")
    ax.plot(sub["n_splits"], sub["t_bat_ms"], "s-", label="batched", color="#1f77b4")
    ax.set_title(net_name, fontweight="bold")
    ax.set_xlabel("n_splits"); ax.set_ylabel("vreme (ms)")
    ax.set_xscale("log", base=2); ax.set_yscale("log")
    ax.grid(True, linestyle=":", alpha=0.5); ax.legend()
    ax = axes[1, col]
    ax.plot(sub["n_splits"], sub["vram_seq_mb"], "o-", label="sequential", color="#d62728")
    ax.plot(sub["n_splits"], sub["vram_bat_mb"], "s-", label="batched", color="#1f77b4")
    ax.set_xlabel("n_splits"); ax.set_ylabel("peak VRAM (MB)")
    ax.set_xscale("log", base=2)
    ax.grid(True, linestyle=":", alpha=0.5); ax.legend()
fig.suptitle("eps_idx = 0, float64, CUDA: sequential naspram batched", fontsize=13)
fig.tight_layout()
fig.savefig("/content/split_batched_vs_sequential.png", bbox_inches="tight")
plt.show()

## 6. Preuzimanje rezultata (`.zip`)

In [ ]:
from zipfile import ZipFile
from google.colab import files

summary.to_csv("/content/batched_vs_sequential_summary.csv", index=False)
pd.DataFrame(rows).to_csv("/content/batched_gelsd_share_big.csv", index=False)

paths = [
    "/content/test_batched_splits_cuda.log",
    "/content/sweep_sequential.csv", "/content/sweep_sequential.log",
    "/content/sweep_batched_small.csv", "/content/sweep_batched_small.log",
    "/content/sweep_batched_big.csv", "/content/sweep_batched_big.log",
    "/content/batched_vs_sequential_summary.csv",
    "/content/batched_gelsd_share_big.csv",
    "/content/split_batched_vs_sequential.png",
]
archive = "/content/gpu_hybrid_batched_splits.zip"
with ZipFile(archive, "w") as output:
    for path in paths:
        if os.path.exists(path):
            output.write(path, arcname=os.path.basename(path))
print("Arhivirani fajlovi:", [os.path.basename(p) for p in paths if os.path.exists(p)])
files.download(archive)